# CP2 · Notebook 04 — Profundidad relativa vs métrica

## Objetivo

Entender **por qué un foundation model monocular no da metros reales** y qué información adicional se necesita.

Vas a:
1. Inspeccionar los valores del depth map de MiDaS y descubrir que son **inverse depth relativo**.
2. Probar una **calibración naïve** lineal contra una asunción razonable (e.g. "el horizonte está a 50 m").
3. Ver dónde la calibración rompe → conclusión: necesitas más info que una imagen.

**Tiempo estimado**: 8 min.

## 1. Recordatorio: qué nos devuelve MiDaS

Cuando ejecutas:
```python
outputs = model(**inputs).predicted_depth
```
obtienes un tensor 2D donde **cada valor es "inverse depth"**:
- **Valor alto** → más cerca de la cámara.
- **Valor bajo** → más lejos.
- **Sin unidades** físicas — los rangos cambian entre imágenes.

Esto se llama **inverse depth normalizado**: el modelo aprende un ordenamiento, no una distancia.

Para convertirlo en algo parecido a metros, necesitas una **transformación afín** $d_m = a / d_{inv} + b$ con 2 parámetros desconocidos $a, b$ — y aquí es donde la cosa se complica.

## 2. Cargar depth maps de 03

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

OUT_DIR = Path('../outputs')
DEPTH_DIR = OUT_DIR / '03_midas_depth'

with open(OUT_DIR / '03_midas_summary.json') as f:
    summary = json.load(f)

# Cargar uno de easy para análisis
easy_entries = [e for e in summary['per_image'] if e['category'] == 'easy']
entry = easy_entries[0]
depth = np.load(DEPTH_DIR / entry['npy_file'])

img_path = Path('../../CP1-carriles/datasets/lanes-subset/easy') / entry['name']
img = np.array(Image.open(img_path).convert('RGB'))

print(f'Imagen: {entry["name"]}  shape img={img.shape}  shape depth={depth.shape}')
print(f'Depth stats:  min={depth.min():.2f}  max={depth.max():.2f}  mean={depth.mean():.2f}')

## 3. Visualizar: inverse depth vs "depth aparente en metros"

Probamos varias funciones de mapeo y vemos cuál "parece" más metros.

In [ ]:
# Si depth = inverse depth, entonces "depth aparente" = 1/depth (con cuidado de no dividir por 0)
eps = 1e-6
apparent_depth = 1.0 / (depth + eps)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
im1 = axes[1].imshow(depth, cmap='plasma')
axes[1].set_title(f'MiDaS output (inverse depth, alto=cerca)\nrange [{depth.min():.1f}, {depth.max():.1f}]')
axes[1].axis('off'); plt.colorbar(im1, ax=axes[1], fraction=0.046)
im2 = axes[2].imshow(apparent_depth, cmap='viridis')
axes[2].set_title(f'1 / output (depth aparente, bajo=cerca)\nrange [{apparent_depth.min():.2f}, {apparent_depth.max():.2f}]')
axes[2].axis('off'); plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout(); plt.show()

## 4. Calibración naïve con 2 puntos

Suposiciones razonables sobre una imagen de autopista típica:
- El **bonete del coche** está a ~2 m de la cámara.
- El **horizonte** (línea donde los carriles convergen) está a ~80 m.

Con 2 puntos podemos ajustar la relación afín $d_m = a / d_{inv} + b$ y obtener algo que parece metros.

**Asumiremos** que el píxel más cerca del bottom-center es el bonete, y el píxel del centro-vertical es el horizonte.

In [ ]:
H, W = depth.shape
# Píxeles asumidos
hood_y, hood_x = int(0.95 * H), W // 2
horizon_y, horizon_x = int(0.55 * H), W // 2

inv_hood    = depth[hood_y, hood_x]
inv_horizon = depth[horizon_y, horizon_x]

print(f'pixel hood:    ({hood_x},{hood_y})    inv_depth = {inv_hood:.3f}   → asumido 2 m')
print(f'pixel horizon: ({horizon_x},{horizon_y}) inv_depth = {inv_horizon:.3f}  → asumido 80 m')

# Modelo: d_m = a / d_inv + b
# 2 = a / inv_hood    + b
# 80 = a / inv_horizon + b
A = np.array([[1 / inv_hood, 1], [1 / inv_horizon, 1]])
y = np.array([2.0, 80.0])
a, b = np.linalg.solve(A, y)
print(f'\nCalibración: d_m = {a:.2f} / d_inv + {b:.2f}')

metric_depth = a / (depth + eps) + b
# Clamp a [0, 200] m para visualización
metric_depth = np.clip(metric_depth, 0, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
im = axes[1].imshow(metric_depth, cmap='viridis_r', vmin=0, vmax=120)
axes[1].plot(hood_x, hood_y, 'r*', markersize=15, label=f'asumido 2 m')
axes[1].plot(horizon_x, horizon_y, 'y*', markersize=15, label=f'asumido 80 m')
axes[1].set_title('Depth en "metros" (calibración 2-puntos)')
axes[1].axis('off')
axes[1].legend(loc='lower right')
plt.colorbar(im, ax=axes[1], fraction=0.046, label='metros')
plt.tight_layout(); plt.show()

## 5. La trampa: la calibración **no transfiere** entre imágenes

Si aplicamos los mismos $(a, b)$ a una imagen distinta, ¿funciona?

In [ ]:
# Tomar una imagen 'hard' (sombras, curvas)
hard_entries = [e for e in summary['per_image'] if e['category'] == 'hard']
entry2 = hard_entries[0]
depth2 = np.load(DEPTH_DIR / entry2['npy_file'])
img2_path = Path('../../CP1-carriles/datasets/lanes-subset/hard') / entry2['name']
img2 = np.array(Image.open(img2_path).convert('RGB'))

metric2 = a / (depth2 + eps) + b
metric2 = np.clip(metric2, 0, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(img2); axes[0].set_title(f'Hard · {entry2["name"]}'); axes[0].axis('off')
im = axes[1].imshow(metric2, cmap='viridis_r', vmin=0, vmax=120)
axes[1].set_title('Mismo (a, b) aplicados — ¿son metros reales?')
axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046)
plt.tight_layout(); plt.show()

print(f'Imagen easy ref:  depth range orig=[{depth.min():.2f}, {depth.max():.2f}]')
print(f'Imagen hard:      depth range orig=[{depth2.min():.2f}, {depth2.max():.2f}]')
print('\nObservación: si los rangos de inverse-depth difieren, los "metros" estimados también.')
print('Eso es el problema fundamental — el modelo no garantiza consistencia de escala entre frames.')

## 6. ¿Qué cambiaría?

Para tener métrica fiable necesitas **información adicional**:

| Fuente | Cómo ayuda |
|--------|------------|
| **Cámara estéreo** | 2 cámaras separadas → disparidad → métrica directa |
| **LiDAR sparse** | Pocos puntos métricos por frame → calibración por imagen |
| **Objeto de tamaño conocido** | Señal vial 60 cm de alto → escala vía geometría |
| **HD Maps** | El sistema sabe a qué distancia están landmarks → reverse-calibrar |
| **IMU + SfM** | Estructura geométrica multi-frame (más para AR / robotaxi) |
| **Modelo métrico** (ZoeDepth, Marigold) | Entrenado con GT métrico KITTI/Hypersim → da metros directos pero generaliza peor OOD |

**Conclusión operativa**: para un sistema real **camera-only**, MiDaS + un objeto de tamaño conocido + temporal smoothing es lo más cercano a métrico sin caer en LiDAR. Tesla hace algo parecido.

## 7. Mini-experimento: ¿la escala es estable entre frames adyacentes?

Comparamos el `mean(depth)` entre las 5 imágenes `easy` (todas son escenarios similares). Si MiDaS fuera escala-consistente entre frames similares, las medias serían parecidas.

In [ ]:
import pandas as pd
rows = []
for e in summary['per_image']:
    rows.append({
        'name': e['name'], 'category': e['category'], 'origin': e['origin'],
        'inv_min': e['depth_min'], 'inv_max': e['depth_max'], 'inv_mean': e['depth_mean'],
    })
df = pd.DataFrame(rows)
for cat in ['easy','medium','hard','challenge']:
    sub = df[df['category'] == cat]
    if len(sub) == 0: continue
    print(f'{cat:10s}  N={len(sub)}  inv_mean range = [{sub["inv_mean"].min():.2f}, {sub["inv_mean"].max():.2f}]  varianza relativa = {sub["inv_mean"].std()/sub["inv_mean"].mean()*100:.0f}%')
print('\nLa varianza relativa indica que el rango del depth no es estable entre frames.')
print('Esto es lo que llamamos "falta de escala". Para planning real es un problema.')

## 8. Cierre del notebook

Lo que te llevas:

1. **MiDaS da inverse depth relativo** — no metros.
2. **Calibración naïve** con asunciones por imagen funciona localmente pero **no transfiere** entre frames.
3. **Para métrica fiable** necesitas información adicional (estéreo, LiDAR sparse, objetos conocidos…).
4. Esto es **limitación geométrica**, no del modelo — un modelo perfectamente preciso seguiría sin saber la escala absoluta sin info adicional.

Ve a `05_depth_anything_v2.ipynb` para ver si DA v2 hace algo distinto sobre las mismas imágenes.